In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import warnings
import sklearn
warnings.filterwarnings('ignore')

In [ ]:
display (os.getcwd())

'/content'

In [ ]:
dt= pd.read_csv("/audi.csv")
dt.head()

FileNotFoundError: [Errno 2] No such file or directory: '/audi.csv'

In [ ]:
len(dt)

In [ ]:
dt.shape

In [ ]:
dt.columns

In [ ]:
dt.info()

In [ ]:
# The numerical features play a big role in this Regression model,
# so it is important to understand well how are they distributed in the Database.

dt.describe()

In [ ]:
dt.isnull().sum()

In [ ]:
dt.dtypes

In [ ]:
pip install pandas_profiling

In [ ]:
import pandas_profiling as pf
display(pf.ProfileReport(dt))

Separate the Price column

In [ ]:
X = dt.iloc[:,[0,1,3,4,5,6,7,8]].values
display (X.shape)
display (X)

store price column in Y

In [ ]:
Y = dt.iloc[:,[2]].values
display (Y.shape)
display (Y)


In [ ]:
display(pd.DataFrame(X).head(5))

In [ ]:
from sklearn.preprocessing import LabelEncoder
le1 = LabelEncoder()
X[:,0] = le1.fit_transform(X[:,0])
le2 = LabelEncoder()
X[:,-4] = le2.fit_transform(X[:,-4])
display (X)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
ct = ColumnTransformer(transformers = [('encoder',OneHotEncoder(),[2])],remainder='passthrough')
X = ct.fit_transform(X)
display (X.shape)
display (pd.DataFrame(X))

In [ ]:
pd.DataFrame(X)

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X = sc.fit_transform(X)
display (pd.DataFrame(X))

In [ ]:
from sklearn.model_selection import train_test_split
(X_train,X_test,Y_train,Y_test) = train_test_split(X,Y,test_size=0.2,random_state=0)
print (X.shape, Y.shape)
print (X_train.shape, Y_train.shape)
print (X_test.shape, Y_test.shape)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
regression = RandomForestRegressor(random_state=0)
regression.fit(X_train,Y_train)
regression

In [ ]:
y_pred = regression.predict(X_test)
y_pred

In [ ]:
print(np.concatenate((y_pred.reshape(len(y_pred),1),Y_test.reshape(len(Y_test),1)),1))

Accuracy and Mean Absolute Error

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error
print  ('R2 Score ', r2_score(Y_test, y_pred))
print  ('Mean Absolute Error', mean_absolute_error(Y_test,y_pred))

Linear Regression Model

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression()
reg.fit(X_train,Y_train)
reg

Prediction with Test Data

In [ ]:
y_pred = reg.predict(X_test)
y_pred

Actual and Predicted Values

In [ ]:
print(np.concatenate((y_pred.reshape(len(y_pred),1),Y_test.reshape(len(Y_test),1)),1))

Accuracy and Mean Absolute Error

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error
print  ('R2 Score ', r2_score(Y_test, y_pred))
print  ('Mean Absolute Error', mean_absolute_error(Y_test,y_pred))

Prediction for complete car data set

In [ ]:
y_pred = reg.predict(X)
display (y_pred)

Actual and predicted data

In [ ]:
result = pd.concat([dt,pd.DataFrame(y_pred)],axis=1)
result

Model Extra Tree Regressor

In [ ]:
from sklearn.ensemble import  ExtraTreesRegressor
ET_Model=ExtraTreesRegressor(n_estimators = 120)
ET_Model.fit(X_train,Y_train)
y_predict=ET_Model.predict(X_test)
from sklearn.metrics import r2_score,mean_absolute_error
print  ('R2 Score ', r2_score(Y_test, y_predict))
print  ('Mean Absolute Error', mean_absolute_error(Y_test,y_predict))

In [ ]:
y_pred = reg.predict(X)
result = pd.concat([dt,pd.DataFrame(y_pred)],axis=1)
print(y_pred)
result

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
n_estimators = [int(x) for x in np.linspace(start = 80, stop = 1500, num = 10)]
max_features = ['auto', 'sqrt']
max_depth = [int(x) for x in np.linspace(6, 45, num = 5)]
min_samples_split = [2, 5, 10, 15, 100]
min_samples_leaf = [1, 2, 5, 10]

# Random Grid

rand_grid={'n_estimators': n_estimators,
               'max_features': max_features,
               'max_depth': max_depth,
               'min_samples_split': min_samples_split,
               'min_samples_leaf': min_samples_leaf}

rf=RandomForestRegressor()

rCV=RandomizedSearchCV(estimator=rf,param_distributions=rand_grid,scoring='neg_mean_squared_error',n_iter=3,cv=3,random_state=42, n_jobs = 1)

In [ ]:
rCV.fit(X_train,Y_train)

In [ ]:
rf_pred=rCV.predict(X_test)
rf_pred

Mean Absolute and Mean squared Error

In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error
print('MAE',mean_absolute_error(Y_test,rf_pred))
print('MSE',mean_squared_error(Y_test,rf_pred))

Model CatBoostRegressor

In [ ]:
pip install CatBoost

In [ ]:
from catboost import CatBoostRegressor
cat = CatBoostRegressor()
print (cat.fit(X_train,Y_train))

In [ ]:
from catboost import CatBoostRegressor
cat=CatBoostRegressor()
print (cat.fit(X_train,Y_train))

In [ ]:
cat_pred=cat.predict(X_test)
display (cat_pred)

In [ ]:
display (r2_score(Y_test,cat_pred))

In [ ]:
import pickle
pickle.dump(cat, open('model.pkl','wb'))

In [ ]:
model=pickle.load(open('model.pkl','rb'))
print (model.predict (X_train))

With this project, we have built a model that can predict with the price of used cars, given a set of features. This information can have an enormous value for both companies and individuals when trying to understand how to estimate the value of a vehicle and, more importantly, the key factors that determine its pricing.